# testing

> A host and a backend with nothing behind either, so the harness can be driven end to end without a model.

Nothing here loads a model, and that is the bargain: what the harness is about is routing,
approval, compaction arithmetic, skill discovery, the activity stream and the tool wrappers,
and a real engine puts gigabytes and minutes in front of all of it while testing none of it.

These were leela's test fixtures. They ship now because `Host` is a thing other applications
are supposed to implement, and the clearest statement of what it requires is a host that
already satisfies it -- `MemHost` is forty lines and a dict.


In [ ]:
#| default_exp testing

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import time
from pathlib import Path

from ramabana.backend import Backend, Usage
from ramabana.chat import Agent
from ramabana.host import Hit, NullHost
from ramabana.models import ModelSpec

In [ ]:
#| export
#: What `FakeBackend` reports itself as, so a status line under a fake model still says
#: something true rather than blank. The small window is deliberate: compaction arithmetic
#: is easier to test against 1000 tokens than against 200,000.
SPEC = ModelSpec('fake', 'fake', 'fake/model', ctx=1000)

#: `ScriptedBackend`'s own spec, kept distinct because it is used for screenshots rather
#: than for tests -- a capture of the IDE should not have "fake" written across the status
#: bar, and its window has to be big enough that compaction does not fire mid-screenshot.
SCRIPTED = ModelSpec('scripted', 'scripted', 'scripted/model', ctx=8000)

In [ ]:
#| export
class MemHost(NullHost):
    "A host whose folders live in a dict, so the file tools can be driven without touching disk."

    def __init__(self, files=None, root='/proj'):
        super().__init__([root])
        self.files, self.root = dict(files or {}), root
        self.ran = []

    def check(self, path, must_exist=False):
        p = Path(path)
        return p if p.is_absolute() else Path(self.root)/p

    def walk(self): return list(self.files)
    def read(self, path): return self.files.get(str(path))
    def text_at(self, path): return self.files.get(str(path), '')

    def write(self, path, text):
        self.files[str(path)] = text
        return str(path)

    def search(self, query, limit=20):
        return [Hit(p, 1, '', t.splitlines()[0]) for p, t in self.files.items() if query in t][:limit]

    @property
    def search_note(self): return 'memory'

    def run_python(self, code):
        self.ran.append(code)
        return 'ok'

In [ ]:
#| export
class FakeBackend(Backend):
    "A backend over a scripted list of replies, so a turn can be driven with no model at all."

    kind = 'fake'

    def __init__(self, spec=SPEC, replies=(), **kw):
        super().__init__(spec, **kw)
        self.replies, self.sent, self.hist_ = list(replies), [], []
        self.spawned = []

    def _start(self): return self
    def _close(self): pass

    def _send(self, msg, **kw):
        self.sent.append(msg)
        self.hist_.append({'role': 'user', 'content': str(msg)})
        out = self.replies.pop(0) if self.replies else '(done)'
        self.hist_.append({'role': 'assistant', 'content': out})
        return out

    def _stream(self, msg, **kw):
        for w in self._send(msg, **kw).split(' '): yield w + ' '

    def _oneshot(self, prompt, sp, max_tokens): return f'ONESHOT:{prompt[:40]}'
    def _usage(self): return Usage(model=self.spec.model_id, input=10, output=5, total=15, turns=1)

    @property
    def hist(self): return self.hist_

    def _replace_hist(self, summary, keep):
        self.hist_ = [{'role': 'user', 'content': summary}] + list(keep)

    def spawn(self, sp='', tools=(), **kw):
        s = FakeBackend(self.spec, replies=['sub answer'], sp=sp, tools=tools, shared=True)
        self.spawned.append(s)
        return s

In [ ]:
#| export
def fake_agent(host=None, replies=(), **kw):
    "An `Agent` whose every job routes to one `FakeBackend`. Returns `(agent, backend)`."
    a = Agent(host or MemHost({'/proj/a.py': 'def a(): pass\n'}), extensions=False, **kw)
    be = FakeBackend(SPEC, replies=replies)
    a._be = lambda job='turn': be
    a._be_or_none = lambda job='turn': be
    return a, be

In [ ]:
#| export
#: What a local Gemma says when it refuses a turn. Real output, kept verbatim, because the
#: whole point of `native` is recognising the shape of this rather than a tidied version.
GEMMA = ('E0000 00:00:1234.567 llm_engine.cc:412] input token IDs exceed the maximum '
         'number of tokens 4096, got 5092\n')


class MutteringBackend(Backend):
    "A backend that fails the way litert does: it prints, returns nothing, and raises nothing."

    kind = 'muttering'

    def __init__(self, spec=None, mode='empty', **kw):
        super().__init__(spec or ModelSpec('gemma-e2b', 'muttering', 'gemma/e2b', ctx=4096), **kw)
        self.mode = mode

    def _start(self): return self
    def _close(self): pass
    def _usage(self): return Usage(model=self.spec.model_id)

    def _mutter(self):
        "What `RishiBackend._native` does for real: keep what the engine said, and report it."
        self.last_native = GEMMA.strip()
        self.problem(f'{self.spec.name}: {GEMMA.strip()}')

    def _send(self, msg, **kw):
        self._mutter()
        return ''

    def _stream(self, msg, **kw):
        self._mutter()
        return iter(())

    def _oneshot(self, prompt, sp, max_tokens):
        self._mutter()
        raise RuntimeError('input too long')

In [ ]:
#| export
class Step:
    """One thing a scripted model does: call a tool, or say something.

    `tool` is a `(name, kwargs)` pair and is called through the agent's real tool list, so
    the activity feed, the approval gate and the file snapshots all run for real -- the only
    fiction is which tool the model decided to call.
    """

    def __init__(self, text='', tool=None, pause=0.0):
        self.text, self.tool, self.pause = text, tool, pause


class ScriptedBackend(Backend):
    """Plays a list of `Step`s, streaming its words one at a time.

    `token_delay` is what makes a screenshot possible: at zero the turn finishes before the
    first repaint and every capture shows a completed answer, which is the one thing a
    streaming screenshot must not show.
    """

    kind = 'scripted'

    #: What a spawned sub-agent answers, keyed by a substring of the question. A fan-out
    #: whose three sub-agents all say the same thing would look like one call in a wig.
    SUB_ANSWERS = {'import': 'three files: backend.py, models.py, fastllm_hitl.py',
                   'compaction': 'chat.py:compact(), fired from _prepare() at the threshold',
                   'shape': 'df is (200, 2); `keep` is an int'}

    def __init__(self, spec=SCRIPTED, steps=(), token_delay=0.02, **kw):
        super().__init__(spec, **kw)
        self.steps, self.token_delay = list(steps), token_delay
        self.hist_, self.sent = [], []

    def _start(self): return self
    def _close(self): pass
    def _oneshot(self, prompt, sp, max_tokens): return 'a one-shot answer'
    def _usage(self): return Usage(model=self.spec.model_id, input=1840, output=96, total=1936,
                                   cached=1200, cost=0.0031, turns=1)

    @property
    def hist(self): return self.hist_

    def _replace_hist(self, summary, keep):
        self.hist_ = [{'role': 'user', 'content': summary}] + list(keep)

    def spawn(self, sp='', tools=(), **kw):
        answers = self.SUB_ANSWERS
        class Sub(ScriptedBackend):
            def _run(self, msg):
                q = str(msg).lower()
                hit = next((v for k, v in answers.items() if k in q), 'nothing found')
                for w in hit.split(' '): yield w + ' '
        return Sub(self.spec, token_delay=0, shared=True)

    def _tool(self, name):
        return next((t for t in self.tools if getattr(t, '__name__', '') == name), None)

    def _run(self, msg):
        "Walk the script, calling tools for real and yielding the words of every text step."
        self.sent.append(msg)
        self.hist_.append({'role': 'user', 'content': str(msg)})
        for s in self.steps:
            if s.pause: time.sleep(s.pause)
            if s.tool:
                name, kw = s.tool
                if (f := self._tool(name)) is not None:
                    self.hist_.append({'role': 'assistant', 'content': f'[{name}]'})
                    self.hist_.append({'role': 'tool', 'content': str(f(**kw))[:400]})
            for w in s.text.split(' '):
                if not w: continue
                if self.token_delay: time.sleep(self.token_delay)
                yield w + ' '

    def _send(self, msg, **kw):
        out = ''.join(self._run(msg))
        self.hist_.append({'role': 'assistant', 'content': out})
        return out

    def _stream(self, msg, **kw):
        out = []
        for w in self._run(msg):
            out.append(w)
            yield w
        self.hist_.append({'role': 'assistant', 'content': ''.join(out)})

## Tests


In [ ]:
h = MemHost({'/proj/a.py': 'def a(): pass\n'})
print('walk  :', h.walk())
print('read  :', h.read('/proj/a.py').strip())
print('search:', h.search('def a'))
h.write('/proj/b.py', 'y = 2\n')
assert h.read('/proj/b.py') == 'y = 2\n'

In [ ]:
a, be = fake_agent(replies=['I looked at it.'])
print(a.ask('what is in a.py?'))
print('sent to the model:', len(be.sent), 'message(s)')
assert be.sent